# EDA Basis

In [ ]:
# General TODOs:

# General TODOs:

# TODO: ...

# General NOTEs:

# NOTE: ...

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

from thesis_project import (
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "eda-basis"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

## Utilities

In [ ]:
import re


def to_seconds(s):
    match = re.match(r"(\d+)([a-zA-Z]+)", s)
    value, unit = match.groups()  # type: ignore
    value = int(value)

    multipliers = {
        "s": 1,
        "min": 60,
        "h": 3600,
    }

    return value * multipliers[unit]

### Plotting

# Main

## Common

In [ ]:
# Basis data
filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
basis_df = pd.read_parquet(config.INT_DIR / TICKER / "data-microstructure" / "basis" / filename)
basis_df = basis_df.astype("float32")
basis_df = basis_df.sort_index()
assert isinstance(basis_df.index, pd.DatetimeIndex)

idx = basis_df.index
tz = idx.tz

BASIS_RESAMPLE_FREQS = [
    "1s",
    "5s",
    "10s",
    "15s",
    "30s",
    "1min",
]

In [ ]:
fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

In [ ]:
# Get filtered relevant dates
start = basis_df.index.min().normalize()
end = basis_df.index.max().normalize()

# Last trading dates of futures
FUT_LAST_TRADING_DATES = pd.to_datetime(fut_meta_df["Last Trading Date"].values)
FUT_LAST_TRADING_DATES = FUT_LAST_TRADING_DATES.tz_localize(tz).normalize()
FUT_LAST_TRADING_DATE_FILTERED = FUT_LAST_TRADING_DATES[
    (FUT_LAST_TRADING_DATES >= start) & (FUT_LAST_TRADING_DATES <= end)
]

# CTD switch dates
CTD_SWITCH_DATES = pd.to_datetime(ctd_switch_df["CTD Switch Date"].values)
CTD_SWITCH_DATES = CTD_SWITCH_DATES.tz_localize(tz).normalize()
CTD_SWITCH_DATES_FILTERED = CTD_SWITCH_DATES[
    (CTD_SWITCH_DATES >= start) & (CTD_SWITCH_DATES <= end)
]

## Distribution

In [ ]:
info = []
for freq in BASIS_RESAMPLE_FREQS:
    basis = basis_df["Basis"].resample(freq).last()
    stats = basis.describe()
    stats["skewness"] = basis.skew()
    stats["kurtosis"] = basis.kurtosis()
    stats.name = f"{freq}"
    info.append(stats)
info = pd.DataFrame(info)
info.index.name = "Resample Frequency"

# info.to_csv(OUT_TAB_DIR / "gross_basis_describe_summary.csv")

display(info)

In [ ]:
# Plot basis distribution
import matplotlib.ticker as mtick

# Dates
start_date = basis_df.index.min()
end_date = basis_df.index.max()

# Plot only the full distribution
for freq in BASIS_RESAMPLE_FREQS[0:1]:
    # Cut tails
    plot_data = basis_df["Basis"].resample(freq).last().dropna()
    upper = plot_data.quantile(0.999)
    lower = plot_data.quantile(0.001)
    mask = plot_data.between(lower, upper)
    plot_data = plot_data[mask]

    # Weights
    weights = np.ones_like(plot_data) / len(plot_data)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(
        plot_data,
        bins=600,
        weights=weights,
        # density=True,
    )

    # Labels
    title = (
        f"Basis Distribution ({freq} Resampled)"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
        f"\nQuantiles 0.1% - 99.9%"
    )
    ax.set_title(title)
    ax.set_xlabel("Basis")
    ax.set_ylabel("Frequency")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.grid(True, alpha=0.2)

    # Stats
    median = plot_data.median()
    mean = plot_data.mean()
    std = plot_data.std()

    # Lines
    ax.axvline(
        median, color="green", linestyle="--", linewidth=0.75, label=f"Median ({median:.4f})"
    )
    ax.axvline(mean, color="red", linestyle="--", linewidth=0.75, label=f"Mean ({mean:.4f})")
    ax.axvline(mean + std, color="orange", linestyle="--", linewidth=0.75, label=r"$\pm$ 1 Std")
    ax.axvline(mean - std, color="orange", linestyle="--", linewidth=0.75)

    ax.legend()

    # fig.savefig(OUT_FIG_DIR / f"basis_distribution_{freq}.png", dpi=300)

    plt.tight_layout()
    plt.show()

## Time Series

In [ ]:
# Plot basis timeseries

# Dates
start_date = basis_df.index.min()
end_date = basis_df.index.max()

# Plot limits
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

for freq in BASIS_RESAMPLE_FREQS[0:1]:
    plot_data = basis_df["Basis"].resample(freq).last().dropna()

    # Reduce number of points to plot
    plot_data = plot_data.resample("1min").last().dropna()

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        plot_data,
        label="Basis",
        linewidth=0.5,
    )

    # Labels
    title = (
        f"Basis Time Series ({freq} Resampled)"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
    ax.set_xlim(start_xlim, end_xlim)

    # Vertical Lines
    for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        ax.axvline(
            d,
            color="black",
            linestyle="--",
            linewidth=0.5,
            label="Futures Last Trading Date" if i == 0 else None,
        )

    # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
    #     ax.axvline(
    #         d,
    #         color='blue',
    #         linestyle='--',
    #         linewidth=0.5,
    #         label='CTD Switch Date' if i == 0 else None
    #     )

    ax.axhline(0, color="red", linestyle="--", linewidth=0.5)

    ax.legend(loc="upper left")

    # fig.savefig(OUT_FIG_DIR / f"basis_timeseries_{freq}.png", dpi=300)

    plt.tight_layout()
    plt.show()

### Day Sample

In [ ]:
# Plot a sample day of basis
assert isinstance(basis_df.index, pd.DatetimeIndex)
year, month, day = "2022", "09", "07"
date = pd.Timestamp(f"{year}-{month}-{day}", tz=basis_df.index.tz)

# Filter plot data
plot_data = basis_df["Basis"]
date_mask = basis_df.index.normalize() == date
start_xlim = date + pd.Timedelta(hours=7, minutes=30)
end_xlim = date + pd.Timedelta(hours=18, minutes=30)
plot_data = basis_df.loc[date_mask]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    plot_data,
    label="Basis",
    linewidth=0.5,
)
title = f"Basis Time Series on {date.strftime('%Y-%m-%d')}"
ax.set_title(title)
ax.set_xlabel("Time")
ax.set_ylabel("Basis")
ax.grid(True, alpha=0.3)
ax.legend()
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=basis_df.index.tz))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=basis_df.index.tz))
ax.set_xlim(start_xlim, end_xlim)
fig.tight_layout()
plt.show()

## Rolling Statistics

In [ ]:
ROLLING_STAT_WINDOWS = {
    "10s": 10,
    "30s": 30,
    "1min": 60,
    "5min": 300,
    "10min": 600,
    "30min": 1800,
    "1h": 3600,
}

ROLLING_STATS = [
    "mean",
    "std",
]

In [ ]:
# Compute rolling statistics for basis
assert isinstance(basis_df.index, pd.DatetimeIndex)

day = basis_df.index.date
s = basis_df["Basis"]
grouped = s.groupby(day)
cols = []
for key, window in ROLLING_STAT_WINDOWS.items():
    w = f"{window}s"

    mean = grouped.transform(lambda x: x.rolling(w).mean())
    std = grouped.transform(lambda x: x.rolling(w).std())

    mean.name = f"Rolling Mean {key}"
    std.name = f"Rolling Std {key}"

    cols.append(mean)
    cols.append(std)

rolling_stats_df = pd.concat(cols, axis=1)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_rolling_stats.parquet"
# rolling_stats_df.to_parquet(path / filename)
rolling_stats_df = pd.read_parquet(path / filename)

In [ ]:
# Plot rolling statistics
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)
resample_freq = "1min"

subset = [
    "10s",
    "1min",
    "10min",
    "1h",
]
filtered = {key: ROLLING_STAT_WINDOWS[key] for key in subset}

n_plots = len(filtered)
cols = min(2, n_plots)
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
axes = axes.flatten()

for ax, (key, stats) in zip(axes, filtered.items()):
    for stat in ROLLING_STATS:
        plot_data = (
            rolling_stats_df[f"Rolling {stat.capitalize()} {key}"]
            .resample(resample_freq)
            .last()
            .dropna()
        )
        ax.plot(
            plot_data,
            label=f"Rolling {stat}",
            linewidth=0.5,
        )
    plot_data = basis_df["Basis"].resample(resample_freq).last().dropna()
    ax.plot(
        plot_data,
        label="Basis",
        linewidth=0.5,
        alpha=0.3,
        color="gray",
    )
    title = f"Rolling window {key}"
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Rolling Statistics")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
    ax.set_xlim(start_xlim, end_xlim)
    for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        ax.axvline(
            d,
            color="black",
            linestyle="--",
            linewidth=0.5,
            label="Futures Last Trading Date" if i == 0 else None,
        )
    # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
    #     ax.axvline(
    #         d,
    #         color='blue',
    #         linestyle='--',
    #         linewidth=0.5,
    #         label='CTD Switch Date' if i == 0 else None
    #     )
    ax.legend(loc="upper left")

    suptitle = (
        f"Basis Rolling Statistics"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    )
    fig.suptitle(suptitle, fontsize=16)

for ax in axes[n_plots:]:
    ax.remove()

plt.show()

### Day sample

In [ ]:
# Plot a sample day of basis rolling statistics
assert isinstance(rolling_stats_df.index, pd.DatetimeIndex)
year, month, day = "2022", "09", "07"
date = pd.Timestamp(f"{year}-{month}-{day}", tz=rolling_stats_df.index.tz)

# FIlter plot data
date_mask = rolling_stats_df.index.normalize() == date
plot_data = rolling_stats_df.loc[date_mask]

start_xlim = date + pd.Timedelta(hours=7, minutes=30)
end_xlim = date + pd.Timedelta(hours=18, minutes=30)

subset = ROLLING_STAT_WINDOWS.keys()
filtered = {key: ROLLING_STAT_WINDOWS[key] for key in subset}

n_plots = len(filtered)
cols = 2
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
axes = axes.flatten()

for ax, (key, stats) in zip(axes, filtered.items()):
    for stat in ROLLING_STATS:
        ax.plot(
            plot_data[f"Rolling {stat.capitalize()} {key}"],
            label=f"Rolling {stat}",
            linewidth=0.5,
        )

    title = f"Rollling window {key}"
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Rolling Statistics")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=tz))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=tz))
    ax.set_xlim(start_xlim, end_xlim)

    axt = ax
    axt.plot(
        basis_df.loc[date_mask, "Basis"],
        label="Basis",
        color="gray",
        alpha=0.5,
        linewidth=0.5,
    )
    # axt.set_ylabel("Basis")
    # axt.grid(False)

    # handles1, labels1 = ax.get_legend_handles_labels()
    # handles2, labels2 = axt.get_legend_handles_labels()
    # ax.legend(handles1 + handles2, labels1 + labels2, loc="upper left")
    ax.legend(loc="upper left")

for ax in axes[n_plots:]:
    ax.remove()

suptitle = f"Basis Rolling Statistics on {date.strftime('%Y-%m-%d')}"
fig.suptitle(suptitle, fontsize=16)
plt.show()

## ACF and PACF

### Basis

In [ ]:
nlags = 300
alpha = 0.05

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

assert isinstance(basis_df.index, pd.DatetimeIndex)


def compute_acf(x):
    x = x.dropna().values

    acf_vals, conf = acf(  # type: ignore
        x, nlags=nlags, fft=True, alpha=alpha
    )
    return acf_vals, conf


def compute_pacf(x):
    x = x.dropna().values

    pacf_vals, conf = pacf(x, nlags=nlags, method="burg", alpha=alpha)
    return pacf_vals, conf


s = basis_df["Basis"].dropna()
grouped = s.groupby(s.index.normalize())  # type: ignore

basis_acf = grouped.apply(compute_acf)
basis_pacf = grouped.apply(compute_pacf)

acf_vals = np.vstack([v[0] for v in basis_acf])
acf_conf = np.stack([v[1] for v in basis_acf])
acf_lower = acf_conf[:, :, 0]
acf_upper = acf_conf[:, :, 1]

pacf_vals = np.vstack([v[0] for v in basis_pacf])
pacf_conf = np.stack([v[1] for v in basis_pacf])
pacf_lower = pacf_conf[:, :, 0]
pacf_upper = pacf_conf[:, :, 1]

cols_val = [f"Lag {i}" for i in range(nlags + 1)]
cols_low = [f"Low CI {i}" for i in range(nlags + 1)]
cols_up = [f"Up CI {i}" for i in range(nlags + 1)]

basis_acf_df = pd.DataFrame(
    np.hstack([acf_vals, acf_lower, acf_upper]),
    index=basis_acf.index,
    columns=cols_val + cols_low + cols_up,
)
basis_acf_df = basis_acf_df.set_index(basis_acf_df.index.date)  # type: ignore
basis_acf_df.index.name = "Date"

basis_pacf_df = pd.DataFrame(
    np.hstack([pacf_vals, pacf_lower, pacf_upper]),
    index=basis_pacf.index,
    columns=cols_val + cols_low + cols_up,
)
basis_pacf_df = basis_pacf_df.set_index(basis_pacf_df.index.date)  # type: ignore
basis_pacf_df.index.name = "Date"

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"

filename = "gross_basis_acf.parquet"
# basis_acf_df.to_parquet(path / filename)
basis_acf_df = pd.read_parquet(path / filename)
display(basis_acf_df.head())

filename = "gross_basis_pacf.parquet"
# basis_pacf_df.to_parquet(path / filename)
basis_pacf_df = pd.read_parquet(path / filename)
display(basis_pacf_df.head())

#### Per Lag Time Series

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

subset = [2]

fig, ax = plt.subplots(figsize=(12, 6))

for lag in subset:
    plot_data = basis_acf_df[f"Lag {lag}"]
    ax.plot(
        plot_data,
        label=f"ACF Lag {lag}",
        linewidth=0.5,
    )
    ax.fill_between(
        plot_data.index,
        basis_acf_df[f"Low CI {lag}"],
        basis_acf_df[f"Up CI {lag}"],
        color="blue",
        alpha=0.1,
    )

    plot_data = basis_pacf_df[f"Lag {lag}"]
    ax.plot(
        plot_data,
        label=f"PACF Lag {lag}",
        linewidth=0.5,
    )
    ax.fill_between(
        plot_data.index,
        basis_pacf_df[f"Low CI {lag}"],
        basis_pacf_df[f"Up CI {lag}"],
        color="orange",
        alpha=0.1,
    )

ax.axhline(0, color="red", linestyle="-", linewidth=0.5)

for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
    ax.axvline(
        d,
        color="black",
        linestyle="--",
        linewidth=0.5,
        label="Futures Last Trading Date" if i == 0 else None,
    )
# for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
#     ax.axvline(
#         d,
#         color="blue",
#         linestyle="--",
#         linewidth=0.5,
#         label="CTD Switch Date" if i == 0 else None,
#     )

title = (
    f"ACF and PACF of Basis"
    f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    f"\nPlotted Lags: {', '.join(str(lag) for lag in subset)}"
)
ax.set_title(title)
ax.set_xlabel("Day")
ax.set_ylabel("Correlation")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1, tz=tz))
ax.set_xlim(start_xlim, end_xlim)
ax.legend()

plt.tight_layout()
plt.show()

#### Average Lags Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ticks = [f"Lag {i}" for i in range(0, nlags + 1)]

plot_data = basis_acf_df.mean()[[f"Lag {i}" for i in range(nlags + 1)]]
low = basis_acf_df.mean()[[f"Low CI {i}" for i in range(nlags + 1)]]
up = basis_acf_df.mean()[[f"Up CI {i}" for i in range(nlags + 1)]]
ax.plot(
    plot_data,
    label="ACF",
    marker="o",
    markersize=4,
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)
ax.fill_between(ticks, low, up, alpha=0.2, label="ACF Confidence Interval")

plot_data = basis_pacf_df.mean()[[f"Lag {i}" for i in range(nlags + 1)]]
low = basis_pacf_df.mean()[[f"Low CI {i}" for i in range(nlags + 1)]]
up = basis_pacf_df.mean()[[f"Up CI {i}" for i in range(nlags + 1)]]
ax.plot(
    plot_data,
    label="PACF",
    marker="o",
    markersize=4,
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)
ax.fill_between(ticks, low, up, alpha=0.2, label="PACF Confidence Interval")

ax.axhline(0, color="red", linestyle="-", linewidth=0.5)
ax.set_title("Average ACF and PACF of Basis")
ax.set_xlabel("Lag")
ax.set_ylabel("Correlation")
ax.grid(True, alpha=0.3)
ax.set_xticks(ticks[::20])
ax.set_xticks(ticks[::5], minor=True)
ax.legend()

# fig.savefig(OUT_FIG_DIR / f"basis_acf_pacf_1s.png", dpi=300)

plt.show()

### Squared Basis

In [ ]:
assert isinstance(basis_df.index, pd.DatetimeIndex)

s = basis_df["Basis"].dropna() ** 2
grouped = s.groupby(s.index.normalize())  # type: ignore

nlags = 300
alpha = 0.05


def compute_acf(x):
    x = x.dropna().values

    if len(x) <= nlags:
        return (np.full(nlags + 1, np.nan), np.full((nlags + 1, 2), np.nan))

    acf_vals, conf = acf(  # type: ignore
        x, nlags=nlags, fft=True, alpha=alpha
    )
    return acf_vals, conf


def compute_pacf(x):
    x = x.dropna().values

    pacf_vals, conf = pacf(x, nlags=nlags, method="burg", alpha=alpha)
    return pacf_vals, conf


sq_basis_acf = grouped.apply(compute_acf)
sq_basis_pacf = grouped.apply(compute_pacf)

acf_vals = np.vstack([v[0] for v in basis_acf])
acf_conf = np.stack([v[1] for v in basis_acf])
acf_lower = acf_conf[:, :, 0]
acf_upper = acf_conf[:, :, 1]

pacf_vals = np.vstack([v[0] for v in basis_pacf])
pacf_conf = np.stack([v[1] for v in basis_pacf])
pacf_lower = pacf_conf[:, :, 0]
pacf_upper = pacf_conf[:, :, 1]

cols_val = [f"Lag {i}" for i in range(nlags + 1)]
cols_low = [f"Low CI {i}" for i in range(nlags + 1)]
cols_up = [f"Up CI {i}" for i in range(nlags + 1)]
sq_basis_acf_df = pd.DataFrame(
    np.hstack([acf_vals, acf_lower, acf_upper]),
    index=basis_acf.index,
    columns=cols_val + cols_low + cols_up,
)
sq_basis_pacf_df = pd.DataFrame(
    np.hstack([pacf_vals, pacf_lower, pacf_upper]),
    index=basis_pacf.index,
    columns=cols_val + cols_low + cols_up,
)

#### Per Lag Time Series

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

subset = range(1, 11)

fig, ax = plt.subplots(figsize=(12, 6))
x = sq_basis_acf_df.index

for lag in subset:
    # acf_vals = sq_basis_acf_df[f"Lag {lag}"]
    # acf_low = sq_basis_acf_df[f"Low CI {lag}"]
    # acf_up = sq_basis_acf_df[f"Up CI {lag}"]
    # ax.plot(
    #     x,
    #     acf_vals,
    #     linewidth=0.5,
    #     label=f"ACF Lag {lag}"
    # )
    # ax.fill_between(
    #     x,
    #     acf_low,
    #     acf_up,
    #     alpha=0.2
    # )

    pacf_vals = sq_basis_pacf_df[f"Lag {lag}"]
    pacf_low = sq_basis_pacf_df[f"Low CI {lag}"]
    pacf_up = sq_basis_pacf_df[f"Up CI {lag}"]
    ax.plot(x, pacf_vals, linewidth=0.5, linestyle="-", label=f"PACF Lag {lag}")
    ax.fill_between(x, pacf_low, pacf_up, alpha=0.2)

ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
ax.set_xlim(start_xlim, end_xlim)

for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
    ax.axvline(
        d,
        color="black",
        linestyle="--",
        linewidth=0.5,
        label="Futures Last Trading Date" if i == 0 else None,
    )
# for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
#     ax.axvline(
#         d,
#         color='green',
#         linestyle='--',
#         linewidth=0.5,
#         label='CTD Switch Date' if i == 0 else None
#     )

ax.axhline(0, color="red", linestyle="-", linewidth=0.5)

title = (
    f"ACF and PACF of Squared Basis"
    f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    f"\nPlotted Lags: ({', '.join(str(lag) for lag in subset)})"
)
ax.set_title(title)
ax.set_xlabel("Day")
ax.set_ylabel("Correlation")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

#### Average Lags Time Series

In [ ]:
ticks = [f"Lag {i}" for i in range(0, nlags + 1)]

fig, ax = plt.subplots(figsize=(12, 6))

plot_data = sq_basis_acf_df.mean()[[f"Lag {i}" for i in range(nlags + 1)]]
low = sq_basis_acf_df.mean()[[f"Low CI {i}" for i in range(nlags + 1)]]
up = sq_basis_acf_df.mean()[[f"Up CI {i}" for i in range(nlags + 1)]]
ax.plot(
    plot_data,
    label="ACF",
    marker="o",
    linestyle="-",
    linewidth=0.5,
)
ax.fill_between(ticks, low, up, alpha=0.2)

plot_data = sq_basis_pacf_df.mean(axis=0)[[f"Lag {i}" for i in range(nlags + 1)]]
low = sq_basis_pacf_df.mean(axis=0)[[f"Low CI {i}" for i in range(nlags + 1)]]
up = sq_basis_pacf_df.mean(axis=0)[[f"Up CI {i}" for i in range(nlags + 1)]]
ax.plot(
    plot_data,
    label="PACF",
    marker="o",
    linestyle="-",
    linewidth=0.5,
)
ax.fill_between(ticks, low, up, alpha=0.2)

ax.set_title("Average ACF and PACF of Squared Basis")
ax.set_xlabel("Lag")
ax.set_ylabel("Correlation")
ax.grid(True, alpha=0.3)
ax.set_xticks(ticks[::15])
ax.set_xticks(ticks[::5], minor=True)
ax.legend()
plt.tight_layout()
plt.show()

## QQ Plot

In [ ]:
from statsmodels.graphics.gofplots import qqplot

n_plots = len(BASIS_RESAMPLE_FREQS)
cols = 3
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
axes = axes.flatten()

for ax, freq in zip(axes, BASIS_RESAMPLE_FREQS):
    plot_data = basis_df["Basis"].resample(freq).last().dropna()
    qqplot(plot_data, line="45", ax=ax)

    ax.set_title(f"Resampled {freq}")
    ax.set_xlabel("Theoretical Quantiles")
    ax.set_ylabel("Sample Quantiles")
    ax.grid(True, alpha=0.3)

suptitle = (
    f"QQ Plots of Basis"
    f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
)
fig.suptitle(suptitle, fontsize=16)
plt.show()

## Stationarity

In [ ]:
def check_stationarity(series, regression):
    adf_stat, adf_pvalue, *_ = adfuller(series, regression=regression)
    kpss_stat, kpss_pvalue, *_ = kpss(series, regression=regression)

    return {
        "ADF": {
            "Statistic": adf_stat,
            "p-value": adf_pvalue,
        },
        "KPSS": {
            "Statistic": kpss_stat,
            "p-value": kpss_pvalue,
        },
    }


STATIONARITY_TESTS = {
    "c": "Constant",
    "ct": "Constant and Trend",
}

### Daily Stationarity

In [ ]:
# Perform day-wise stationarity tests on basis
assert isinstance(basis_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

results = []

for date, group in basis_df.groupby(basis_df.index.normalize()):
    row = {"Date": date.date()}
    basis = group["Basis"]

    for freq in BASIS_RESAMPLE_FREQS:
        resampled = basis if freq == "1s" else basis.resample(freq).last()
        resampled = resampled.dropna()

        for reg, reg_name in STATIONARITY_TESTS.items():
            out = check_stationarity(resampled, reg)
            row.update(
                {
                    f"ADF Statistic {freq} {reg_name}": out["ADF"]["Statistic"],
                    f"ADF p-value {freq} {reg_name}": out["ADF"]["p-value"],
                    f"KPSS Statistic {freq} {reg_name}": out["KPSS"]["Statistic"],
                    f"KPSS p-value {freq} {reg_name}": out["KPSS"]["p-value"],
                }
            )

    results.append(row)

results_df = pd.DataFrame.from_records(results).set_index("Date")

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_stationarity_tests.parquet"

# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

#### Summary

In [ ]:
summary = []

for reg, reg_name in STATIONARITY_TESTS.items():
    for freq in BASIS_RESAMPLE_FREQS:
        adf_col = f"ADF p-value {freq} {reg_name}"
        kpss_col = f"KPSS p-value {freq} {reg_name}"
        adf_ratio = (results_df[adf_col] <= 0.05).mean()
        kpss_ratio = (results_df[kpss_col] > 0.05).mean()
        summary.append(
            {
                "Regression": reg_name,
                "Frequency": freq,
                "ADF Stationary Ratio": f"{adf_ratio * 100:.2f} %",
                "KPSS Stationary Ratio": f"{kpss_ratio * 100:.2f} %",
            }
        )

summary_df = (
    pd.DataFrame(summary)
    .sort_values(
        by=["Regression", "Frequency"],
        key=lambda x: x.map(lambda s: to_seconds(s) if s in BASIS_RESAMPLE_FREQS else s),
    )
    .set_index(["Regression", "Frequency"])
)

# summary_df.to_csv(OUT_TAB_DIR / "gross_basis_stationarity_summary.csv")

display(summary_df)

#### p-value Distribution

In [ ]:
import numpy as np
from matplotlib import ticker as mtick

bins = np.linspace(0, 1, 101)

for reg, reg_name in STATIONARITY_TESTS.items():
    n_plots = len(BASIS_RESAMPLE_FREQS)
    cols = 3
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for freq, ax in zip(BASIS_RESAMPLE_FREQS, axes):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            weights = np.ones_like(plot_data) / len(plot_data)
            ax.hist(
                plot_data,
                bins=bins,  # type: ignore
                alpha=0.5,
                label=f"{test}",
                weights=weights,
            )

        ax.set_title(f"Resampled {freq}")
        ax.set_xlabel("p-value")
        ax.set_ylabel("Frequency")
        ax.set_xticks(bins[::5])
        ax.set_xticks(bins[::1], minor=True)
        ax.set_xlim(0, 0.25)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    suptitle = (
        f"Basis Daily Stationarity Tests"
        f"\np-value Distributions"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(OUT_FIG_DIR / f"basis_stationarity_pvalue_distribution_{reg_name.replace(' ', '_').lower()}.png", dpi=300)

    plt.show()

#### p-value Time Series

In [ ]:
n_plots = len(BASIS_RESAMPLE_FREQS)
cols = 3
rows = (n_plots + cols - 1) // cols

for reg, reg_name in STATIONARITY_TESTS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for ax, freq in zip(axes, BASIS_RESAMPLE_FREQS):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            ax.plot(
                plot_data,
                label=f"{test} p-value",
                linewidth=0.5,
                alpha=0.7,
            )

        ax.axhline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        # for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        #     ax.axvline(
        #         d,
        #         color='black',
        #         linestyle='--',
        #         linewidth=0.5,
        #         label='Futures Last Trading Date' if i == 0 else None
        #     )
        title = f"Resampling Frequency: {freq}"
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("p-value")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=6, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

    suptitle = (
        f"Basis Daily Stationarity Tests"
        f"\np-value Time Series"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    fig.savefig(
        OUT_FIG_DIR
        / f"basis_stationarity_pvalue_timeseries_{reg_name.replace(' ', '_').lower()}.png",
        dpi=300,
    )

    plt.show()

### Intraday Stationarity

In [ ]:
WINDOW = "1h"
WINDOW_TD = pd.Timedelta(WINDOW)
STEP = "1h"
STEP_TD = pd.Timedelta(STEP)

# For lower frequencies we have too little data per hour
subset = [
    "1s",
    "5s",
    "10s",
    "15s",
]
filtered = [freq for freq in BASIS_RESAMPLE_FREQS if freq in subset]

In [ ]:
import gc

from joblib import Parallel, delayed

# from tqdm import
from tqdm.autonotebook import tqdm

MIN_OBS = 10
N_JOBS = -1

warnings.filterwarnings("ignore", category=InterpolationWarning)

assert isinstance(basis_df.index, pd.DatetimeIndex)

basis_series = basis_df["Basis"].sort_index()
resampled_data = {}
for freq in filtered:
    if freq == "1s":
        s = basis_series
    else:
        s = basis_series.resample(freq).last()

    resampled_data[freq] = s.dropna()

windows = []

grouped = basis_series.groupby(basis_series.index.normalize())  # type: ignore

for d, group in grouped:
    day_index = group.index

    if len(day_index) == 0:
        continue

    day_start = day_index.min().floor("1h")
    day_end = day_index.max().ceil("1h")

    current = day_start
    while current + WINDOW_TD <= day_end:
        if (current + WINDOW_TD) > day_index.max():
            break
        windows.append((current, current + WINDOW_TD))
        current += STEP_TD


def process_window(w_start, w_end):
    row = {"Date": w_start}

    for freq, series in resampled_data.items():
        mask = (series.index >= w_start) & (series.index < w_end)
        s = series.loc[mask]

        if len(s) < MIN_OBS:
            continue

        values = np.asarray(s)

        for reg, reg_name in STATIONARITY_TESTS.items():
            try:
                out = check_stationarity(values, reg)

                row[f"ADF Statistic {freq} {reg_name}"] = out["ADF"]["Statistic"]
                row[f"ADF p-value {freq} {reg_name}"] = out["ADF"]["p-value"]
                row[f"KPSS Statistic {freq} {reg_name}"] = out["KPSS"]["Statistic"]
                row[f"KPSS p-value {freq} {reg_name}"] = out["KPSS"]["p-value"]

            except Exception:
                # avoid breaking parallel loop
                continue

    del s, values

    return row


results = Parallel(n_jobs=N_JOBS, backend="threading", batch_size="auto")(
    delayed(process_window)(w_start, w_end)
    for (w_start, w_end) in tqdm(windows, desc="Processing windows", unit="window", colour="black")
)

results_df = pd.DataFrame(results)  # type: ignore

if not results_df.empty:
    results_df.set_index("Date", inplace=True)
    results_df.sort_index(inplace=True)

for col in results_df.columns:
    results_df[col] = results_df[col].astype(np.float32)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = f"gross_basis_stationarity_tests_freq_{WINDOW.lower()}_w_{STEP.lower()}.parquet"

# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

#### Summary

In [ ]:
summary = []

for reg, reg_name in STATIONARITY_TESTS.items():
    for freq in filtered:
        adf_col = f"ADF p-value {freq} {reg_name}"
        kpss_col = f"KPSS p-value {freq} {reg_name}"
        adf_val = results_df[adf_col]
        kpss_val = results_df[kpss_col]

        adf_grouped = adf_val.groupby(adf_val.index.hour)  # type: ignore
        kpss_grouped = kpss_val.groupby(kpss_val.index.hour)  # type: ignore

        for hour in range(24):
            if hour not in adf_grouped.groups or hour not in kpss_grouped.groups:
                continue

            adf_hour = adf_grouped.get_group(hour)
            kpss_hour = kpss_grouped.get_group(hour)

            adf_ratio = (adf_hour <= 0.05).mean()
            kpss_ratio = (kpss_hour > 0.05).mean()

            summary.append(
                {
                    "Regression": reg_name,
                    "Frequency": freq,
                    "Hour": hour,
                    "ADF Stationary Ratio": f"{adf_ratio * 100:.2f} %",
                    "KPSS Stationary Ratio": f"{kpss_ratio * 100:.2f} %",
                }
            )

summary_df = (
    pd.DataFrame(summary)
    .sort_values(
        by=["Regression", "Frequency", "Hour"],
        key=lambda x: x.map(lambda s: to_seconds(s) if s in BASIS_RESAMPLE_FREQS else s),
    )
    .set_index(["Regression", "Frequency", "Hour"])
)

# summary_df.to_csv(OUT_TAB_DIR / "gross_basis_stationarity_summary.csv")

display(summary_df.head(36))

#### p-value Distribution

#### p-value Time Series

## Regime Switch

In [ ]:
from statsmodels.tsa.stattools import zivot_andrews


def check_breaks(series, regression):
    results = zivot_andrews(series, regression=regression)

    za_stat = results[0]
    za_pvalue = results[1]
    za_bpidx = results[-1]

    return {
        "ZA": {
            "Statistic": za_stat,
            "p-value": za_pvalue,
            "Break Point Index": za_bpidx,
        },
    }


STATIONARITY_TESTS = {
    "c": "Constant",
    "ct": "Constant and Trend",
}

### Daily Structural Breaks

In [ ]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=InterpolationWarning)

N_JOBS = -1

basis_series = basis_df["Basis"].sort_index()

resampled_data = {}
for freq in BASIS_RESAMPLE_FREQS:
    if freq == "1s":
        s = basis_series
    else:
        s = basis_series.resample(freq).last()

    resampled_data[freq] = s.dropna()

assert isinstance(basis_series.index, pd.DatetimeIndex)
grouped = list(basis_series.groupby(basis_series.index.normalize()))


def process_day(date, group_index):
    row = {"Date": date.date()}

    start = group_index.min()
    end = group_index.max()

    for freq, series in resampled_data.items():
        mask = (series.index >= start) & (series.index <= end)
        s = series.loc[mask]

        if len(s) < 10:
            continue

        values = s.values

        for reg, reg_name in STATIONARITY_TESTS.items():
            try:
                out = check_breaks(values, reg)

                row[f"ZA Statistic {freq} {reg_name}"] = out["ZA"]["Statistic"]
                row[f"ZA p-value {freq} {reg_name}"] = out["ZA"]["p-value"]
                row[f"ZA Break Point Index {freq} {reg_name}"] = out["ZA"]["Break Point Index"]

            except Exception as e:
                warnings.warn(
                    f"Error processing date {date} with frequency {freq} and regression {reg_name}: {e}"
                )
                continue

    del s, values
    gc.collect()

    return row


results = Parallel(n_jobs=N_JOBS, backend="threading", batch_size="auto")(
    delayed(process_day)(date, group.index)
    for date, group in tqdm(grouped, desc="Processing days", unit="day", colour="black")
)


results_df = (
    pd.DataFrame(results)  # type: ignore
    .set_index("Date")
    .sort_index()
)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_stationarity_break_tests.parquet"

# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

#### Summary

In [ ]:
summary = []

for reg, reg_name in STATIONARITY_TESTS.items():
    for freq in BASIS_RESAMPLE_FREQS:
        za_col = f"ZA p-value {freq} {reg_name}"
        za_ratio = (results_df[za_col] <= 0.05).mean()
        summary.append(
            {
                "Regression": reg_name,
                "Frequency": freq,
                "ZA Stationary Ratio": f"{za_ratio * 100:.2f} %",
            }
        )

summary_df = (
    pd.DataFrame(summary)
    .sort_values(
        by=["Regression", "Frequency"],
        key=lambda x: x.map(lambda s: to_seconds(s) if s in BASIS_RESAMPLE_FREQS else s),
    )
    .set_index(["Regression", "Frequency"])
)

# summary_df.to_csv(OUT_TAB_DIR / "gross_basis_stationarity_summary.csv")

display(summary_df)

#### p-value Distribution

In [ ]:
import numpy as np
from matplotlib import ticker as mtick

bins = np.linspace(0, 1, 101)

for reg, reg_name in STATIONARITY_TESTS.items():
    n_plots = len(BASIS_RESAMPLE_FREQS)
    cols = 3
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for freq, ax in zip(BASIS_RESAMPLE_FREQS, axes):
        for test in ["ZA"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            weights = np.ones_like(plot_data) / len(plot_data)
            ax.hist(
                plot_data,
                bins=bins,  # type: ignore
                alpha=0.5,
                label=f"{test}",
                weights=weights,
            )

        ax.set_title(f"Resampled {freq}")
        ax.set_xlabel("p-value")
        ax.set_ylabel("Frequency")
        ax.set_xticks(bins[::5])
        ax.set_xticks(bins[::1], minor=True)
        ax.set_xlim(0, 0.25)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    suptitle = (
        f"Basis Daily Stationarity Tests with Breaks"
        f"\np-value Distributions"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(OUT_FIG_DIR / f"basis_stationarity_pvalue_distribution_{reg_name.replace(' ', '_').lower()}.png", dpi=300)

    plt.show()

#### p-value Time Series

In [ ]:
start_date = results_df.index.min()
end_date = results_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

n_plots = len(BASIS_RESAMPLE_FREQS)
cols = 3
rows = (n_plots + cols - 1) // cols

for reg, reg_name in STATIONARITY_TESTS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for ax, freq in zip(axes, BASIS_RESAMPLE_FREQS):
        for test in ["ZA"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            ax.plot(
                plot_data,
                label=f"{test} p-value",
                linewidth=0.5,
                alpha=0.7,
            )

        ax.axhline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")

        for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
            ax.axvline(
                d,
                color="black",
                linestyle="--",
                linewidth=0.5,
                label="Futures Last Trading Date" if i == 0 else None,
            )

        # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
        #     ax.axvline(
        #         d,
        #         color="green",
        #         linestyle="--",
        #         linewidth=0.5,
        #         label="CTD Switch Date" if i == 0 else None,
        #     )

        title = f"Resampling Frequency: {freq}"
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("p-value")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=6, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

    suptitle = (
        f"Basis Daily Stationarity Tests with Breaks"
        f"\np-value Time Series"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(
    #     OUT_FIG_DIR
    #     / f"basis_stationarity_pvalue_timeseries_{reg_name.replace(' ', '_').lower()}.png",
    #     dpi=300,
    # )

    plt.show()

#### Breakpoints

In [ ]:
import numpy as np
from matplotlib import ticker as mtick

# bins = np.linspace(0, 1, 101)

for reg, reg_name in STATIONARITY_TESTS.items():
    n_plots = len(BASIS_RESAMPLE_FREQS)
    cols = 3
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for freq, ax in zip(BASIS_RESAMPLE_FREQS, axes):
        for test in ["ZA"]:
            plot_data = results_df[f"{test} Break Point Index {freq} {reg_name}"]
            weights = np.ones_like(plot_data) / len(plot_data)
            ax.hist(
                plot_data,
                bins="fd",  # type: ignore
                alpha=0.5,
                label=f"{test}",
                # weights=weights,
            )

        ax.set_title(f"Resampled {freq}")
        ax.set_xlabel("p-value")
        ax.set_ylabel("Frequency")
        # ax.set_xticks(bins[::5])
        # ax.set_xticks(bins[::1], minor=True)
        # ax.set_xlim(0, 0.25)
        # ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        # ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        # ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    suptitle = (
        f"Basis Daily Stationarity Tests with Breaks"
        f"\np-value Distributions"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(OUT_FIG_DIR / f"basis_stationarity_pvalue_distribution_{reg_name.replace(' ', '_').lower()}.png", dpi=300)

    plt.show()

## Boxplots

### Utilities

In [ ]:
import os
from typing import Optional


def plot_grouped_boxplot(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    order: list,
    tick_labels: list,
    showfliers: bool = False,
    title: Optional[str] = None,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    tick_positions: Optional[list] = None,
    save_path: Optional[str] = None,
):
    plot_data = [df[df[group_col] == val][value_col].dropna().values for val in order]

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.boxplot(
        plot_data,
        tick_labels=tick_labels if tick_positions is None else None,
        showfliers=showfliers,
    )

    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)

    if tick_positions is not None:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([tick_labels[i - 1] for i in tick_positions])

    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()

In [ ]:
# Save path
save_path = None
SAVE = False

# Data copy for boxplot purposes
boxplot_df = basis_df.copy()
boxplot_df["weekday_num"] = boxplot_df.index.dayofweek

# Year
boxplot_df["year"] = boxplot_df.index.year

# Hour of day
boxplot_df["hour"] = boxplot_df.index.hour
hourly_order = list(range(boxplot_df["hour"].min(), boxplot_df["hour"].max() + 1))

# Day of week
boxplot_df["weekday"] = boxplot_df.index.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
boxplot_df["weekday"] = pd.Categorical(
    boxplot_df["weekday"], categories=weekday_order, ordered=True
)

# Week of year
boxplot_df["week_of_year"] = boxplot_df.index.isocalendar().week
weekly_order = sorted(boxplot_df["week_of_year"].unique())

# Month of year
boxplot_df["month"] = boxplot_df.index.month
boxplot_df["month_name"] = boxplot_df.index.strftime("%B")
monthly_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
boxplot_df["month_name"] = pd.Categorical(
    boxplot_df["month_name"], categories=monthly_order, ordered=True
)

### Hour of Day

In [ ]:
title_base = f"Basis Distribution by Hour of Day\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_bp_by_hour_{'w' if showfliers else 'wo'}_outliers.png")
        if SAVE
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="hour",
        order=hourly_order,
        tick_labels=[f"{h:02d}:00" for h in hourly_order],
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Hour of Day",
        save_path=save_path,
    )

### Day of the Week

In [ ]:
title_base = f"Basis Distribution by Weekday\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_weekday_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="weekday",
        order=weekday_order,
        tick_labels=weekday_order,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Day of Week",
        save_path=save_path,
    )

### Week of Year

In [ ]:
title_base = f"Basis Distribution by Week of Year\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

tick_positions = [i for i, val in enumerate(weekly_order, start=1) if val % 4 == 0]
tick_labels = [f"W{val:02d}" for val in weekly_order]

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_week_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="week_of_year",
        order=weekly_order,
        tick_labels=tick_labels,
        tick_positions=tick_positions,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Week of Year",
        save_path=save_path,
    )

# Split by year
for year in boxplot_df["year"].unique():
    year_df = boxplot_df[boxplot_df["year"] == year]
    year_weekly_order = sorted(year_df["week_of_year"].unique())
    year_tick_positions = [i for i, val in enumerate(year_weekly_order, start=1) if val % 4 == 0]
    year_tick_labels = [f"W{val:02d}" for val in year_weekly_order]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_by_week_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )
        plot_grouped_boxplot(
            df=year_df,
            value_col="Basis",
            group_col="week_of_year",
            order=year_weekly_order,
            tick_labels=year_tick_labels,
            tick_positions=year_tick_positions,
            showfliers=showfliers,
            title=f"Basis Distribution by Week of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis",
            xlabel="Week of Year",
            save_path=save_path,
        )

### Month of Year

In [ ]:
title_base = f"Basis Distribution by Month of Year\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_month_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="month_name",
        order=monthly_order,
        tick_labels=monthly_order,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Month of Year",
        save_path=save_path,
    )

# Split by year
for year in boxplot_df["year"].unique():
    year_df = boxplot_df[boxplot_df["year"] == year]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_by_month_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )
        plot_grouped_boxplot(
            df=year_df,
            value_col="Basis",
            group_col="month_name",
            order=monthly_order,
            tick_labels=monthly_order,
            showfliers=showfliers,
            title=f"Basis Distribution by Month of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis",
            xlabel="Month of Year",
            save_path=save_path,
        )

### Contract Month

In [ ]:
# TODO: Add basis boxplot by grouping data for month/week of the future contract